# Phase 2 Step 3 — AI-ify generation (Qwen 2.5 7B Instruct)

Runs on Kaggle's free GPU. Reads the input dataset pushed by `push_kaggle.py` (a JSONL of
`{id, human_text, style}` rows), rewrites each `human_text` in typical AI-assistant style with
Qwen 2.5 7B Instruct, and writes `{id, ai_text}` pairs to `/kaggle/working/output.jsonl` —
flushed incrementally (not buffered to the end), so a killed/interrupted session still leaves
partial results that `pull_kaggle.py` can report on. No DB credentials of any kind reach this
notebook — only text in, text out.

In [ ]:
import subprocess, sys

# Check the GPU's compute capability via nvidia-smi BEFORE importing torch at
# all. Confirmed on a real run: Kaggle's current base image ships torch 2.10
# (cu128), which has dropped Pascal (sm_60) support entirely -- this is not
# something our own pip install caused, it's baked into the base image. A
# Tesla P100 (sm_60) is still a real possible accelerator Kaggle can hand out,
# so pin a torch version known to still support it whenever we see an old
# arch, rather than trusting whatever's preinstalled.
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("nvidia-smi:", smi.stdout.strip() or smi.stderr.strip())

compute_cap = None
if smi.returncode == 0 and smi.stdout.strip():
    compute_cap = smi.stdout.strip().split(",")[-1].strip()

OLD_ARCH_CAPS = {"6.0", "6.1", "5.0", "5.2"}  # Pascal/Maxwell -- dropped in recent torch wheels
if compute_cap in OLD_ARCH_CAPS:
    print(f"GPU compute capability {compute_cap} is an older arch -- pinning a torch "
          f"build known to still support it (2.3.1+cu121) instead of the preinstalled one.")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch==2.3.1", "--index-url", "https://download.pytorch.org/whl/cu121"],
        check=True,
    )
    # The preinstalled torchvision/torchaudio are built against the base
    # image's original (much newer) torch and break once torch is downgraded
    # -- confirmed the hard way: torchvision's _meta_registrations.py calls
    # torch.library.register_fake, which doesn't exist in torch 2.3.1,
    # raising AttributeError deep inside transformers' (transitive, vision-
    # only) import chain. This notebook only does text generation, so just
    # remove torchvision/torchaudio rather than version-matching a third package.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"], check=False)
else:
    print(f"GPU compute capability {compute_cap!r} looks modern enough -- keeping preinstalled torch.")

# Pinned to an exact version known to work with Qwen2.5 on torch 2.3.1 --
# confirmed the hard way: an unbounded "transformers>=4.46" resolved to a
# version whose Qwen2 modeling code assumes newer torch APIs, which raised a
# ModuleNotFoundError for 'Qwen2ForCausalLM' when torch was pinned down to
# 2.3.1 for GPU-arch compatibility (the real ImportError gets masked by
# transformers' lazy-module-loading wrapper as a generic "could not import").
# No bitsandbytes/quantization here -- 8-bit quantization was tested and
# rejected (see run log): Pascal has no int8 tensor cores, so int8 added
# dequantization overhead without any compute speedup on this GPU. Moving to
# a smaller model (Qwen2.5-3B) in plain fp16 instead.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transformers==4.46.3", "accelerate==0.34.2"],
    check=True,
)

import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    cap = torch.cuda.get_device_capability(0)
    sm = f"sm_{cap[0]}{cap[1]}"
    supported = torch.cuda.get_arch_list()
    print("Compute capability:", sm, "| torch supports:", supported)
    if not any(sm in a for a in supported):
        raise RuntimeError(
            f"GPU compute capability {sm} still not in this torch build's supported archs "
            f"{supported} after attempting a compatible pin. Stop here rather than silently "
            "falling back to CPU for a 7B model -- needs a different torch/cuda combination."
        )
else:
    raise RuntimeError("No CUDA GPU visible -- check enable_gpu in kernel-metadata.json and Kaggle GPU quota.")

import transformers
print("transformers version:", transformers.__version__)
# Fail loudly with the REAL underlying error (not transformers' masked
# "Could not import module" message) if Qwen2 support is somehow still broken.
from transformers.models.qwen2 import modeling_qwen2  # noqa: F401
print("Qwen2 modeling module imports cleanly.")

In [ ]:
import glob, json, os

# Kaggle mounts attached datasets under /kaggle/input/datasets/<owner>/<slug>/
# (confirmed by directly listing /kaggle/input on a real run -- NOT the flat
# /kaggle/input/<slug>/ layout some older docs describe). Glob recursively so
# this keeps working regardless of which layout a given kernel environment uses.
candidates = glob.glob('/kaggle/input/**/input.jsonl', recursive=True)
assert candidates, 'No input.jsonl found under /kaggle/input/ -- was the dataset attached to this kernel?'
input_path = candidates[0]
print('Reading', input_path)

rows = []
with open(input_path) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print(f'Loaded {len(rows)} rows')

In [ ]:
# Resume support: if a previous attempt on this same kernel already wrote a
# partial output.jsonl (e.g. this cell was re-run after an interruption),
# skip ids already done.
OUTPUT_PATH = '/kaggle/working/output.jsonl'
done_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                done_ids.add(json.loads(line)['id'])
print(f'{len(done_ids)} rows already done (resuming past these)')

pending = [r for r in rows if r['id'] not in done_ids]
print(f'{len(pending)} rows left to generate')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Switched from Qwen2.5-7B to Qwen2.5-3B-Instruct: 7B (fp16, ~14GB weights)
# left almost no headroom on this 16GB Pascal P100 for reliable batching, and
# 8-bit quantization was tested and rejected (Pascal has no int8 tensor
# cores, so it added dequant overhead without any speedup -- confirmed by a
# bitsandbytes run taking notably *longer* than plain fp16). 3B's much
# smaller footprint (~6GB fp16) should eliminate OOM risk entirely and give a
# real per-token compute speedup on the same hardware. Correctness (padding,
# EOS behavior) was already solved on the 7B runs -- this is purely a speed test.
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype='auto', device_map={'': 0})
model.eval()
print('Model loaded (Qwen2.5-3B-Instruct, fp16, device_map={\'\': 0}).')

In [ ]:
AIIFY_INSTRUCTION = (
    "Rewrite the following text in typical AI-assistant style: uniform sentence lengths, "
    "transition words, generic vocabulary (delve, leverage, robust, crucial, seamless), "
    "parallel triads, hedging, symmetric paragraphs. Preserve the meaning, facts, names, and "
    "numbers exactly. Output only the rewritten text, nothing else.\n\nText:\n{text}"
)

# BATCH_SIZE=1 deliberately -- this test isolates model size as the only
# variable (no batching complexity, no OOM risk to muddy the comparison).
BATCH_SIZE = 1
MAX_NEW_TOKENS = 1024


def build_prompt(text: str) -> str:
    messages = [{'role': 'user', 'content': AIIFY_INSTRUCTION.format(text=text)}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(batch_rows):
    prompts = [build_prompt(r['human_text']) for r in batch_rows]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    results = []
    input_len = inputs['input_ids'].shape[1]  # safe to slice uniformly: left-padding
    for i, row in enumerate(batch_rows):        # keeps every row's real content right-aligned
        gen_tokens = out[i][input_len:]
        text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        results.append({'id': row['id'], 'ai_text': text})
    return results

In [ ]:
import time

start = time.time()
with open(OUTPUT_PATH, 'a') as out_f:
    for batch_start in range(0, len(pending), BATCH_SIZE):
        batch = pending[batch_start:batch_start + BATCH_SIZE]
        try:
            results = generate_batch(batch)
        except Exception as exc:
            print(f'Batch at offset {batch_start} failed, skipping (will show as missing in pull_kaggle.py): {exc}')
            results = None
        finally:
            # Confirmed the hard way: NOT clearing the cache after a failed
            # (esp. OOM) batch let fragmentation compound across the run --
            # once the first OOM hit, nearly every subsequent batch failed too.
            torch.cuda.empty_cache()
        if results is None:
            continue
        for r in results:
            out_f.write(json.dumps(r) + '\n')
        out_f.flush()  # write incrementally -- never buffer all results to end of run
        os.fsync(out_f.fileno())
        done_so_far = batch_start + len(batch)
        elapsed = time.time() - start
        rate = done_so_far / elapsed if elapsed > 0 else 0
        eta_min = (len(pending) - done_so_far) / rate / 60 if rate > 0 else float('nan')
        print(f'{done_so_far}/{len(pending)} done ({rate:.2f} rows/s, ETA {eta_min:.1f} min)')

print('Generation complete.')

In [ ]:
with open(OUTPUT_PATH) as f:
    final_count = sum(1 for line in f if line.strip())
print(f'output.jsonl has {final_count} rows (of {len(rows)} requested in this chunk).')